# TTM-R3 BTCUSDT 30m logReturn — Multivariate Fine-Tuning + Walk-Forward Evaluation

**Model**: [ibm-research/ttm-r3](https://huggingface.co/ibm-research/ttm-r3)  
**Repo**: [zaynety7/granite-tsfm](https://github.com/zaynety7/granite-tsfm)  

### Key TTM-R3 features used
- `mode="mix_channel"` + `forecast_channel_mixing=True` — cross-channel multivariate decoding
- `conditional_columns` — **12** selected Z-score features passed as auxiliary channels
- `prediction_filter_length` — trim native prediction_length output to 6 steps (30 min) at inference
- Quantile head (`loss="pinball"`, quantiles [0.05,0.1,0.25,0.5,0.75,0.9,0.95])
- Walk-forward in-sample / out-of-sample split with rolling re-evaluation

### Channel layout (N=13 total)
| idx | column | role |
| --- | ------ | ---- |
| 0 | `cm_logret` | **target** (loss computed here only) |
| 1 | `basis_cm_spot_z3` | conditional |
| 2 | `basis_compression_z24` | conditional |
| 3 | `basis_from_trades_z3` | conditional |
| 4 | `cm_cvd_acceleration_z24` | conditional |
| 5 | `cm_cvd_change_rate_z3` | conditional |
| 6 | `cm_cvd_cum_z3` | conditional |
| 7 | `cm_vwap_dist_cm_z24` | conditional |
| 8 | `cvd_diff_cm_spot_z3` | conditional |
| 9 | `premium_close_z3` | conditional |
| 10 | `spot_cvd_acceleration_z12` | conditional |
| 11 | `spot_cvd_change_rate_z3` | conditional |
| 12 | `spot_cvd_cum_z3` | conditional |

### Evaluation framework (from tradeTest.md)
- **Model layer**: Pinball Loss per quantile, PICP (80%/90% PI), Quantile Crossing rate, Calibration
- **Strategy layer**: Sharpe, Sortino, Max Drawdown, Hit Rate × Profit Factor, Signal Coverage

In [ ]:
# ── Cell 1 : Install dependencies ──────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('git+https://github.com/zaynety7/granite-tsfm.git')
pip('transformers>=4.40.0', 'datasets', 'accelerate', 'einops')
pip('requests')
pip('plotly', 'kaleido')

In [ ]:
import os, warnings, gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

from tsfm_public import TinyTimeMixerForPrediction, TimeSeriesPreprocessor
from tsfm_public.toolkit.dataset import ForecastDFDataset

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1  Data Download & Feature Engineering

In [ ]:
import requests, time
from datetime import datetime, timezone

BASE_FAPI = 'https://fapi.binance.com'
BASE_API  = 'https://api.binance.com'

def fetch_klines(base_url, endpoint, symbol, start_ms, end_ms, interval='5m'):
    records = []
    url = f'{base_url}{endpoint}'
    cur = start_ms
    while cur < end_ms:
        resp = requests.get(url, params=dict(
            symbol=symbol, interval=interval,
            startTime=cur, endTime=end_ms, limit=1500), timeout=30)
        resp.raise_for_status()
        data = resp.json()
        if not data: break
        records.extend(data)
        cur = int(data[-1][0]) + 1
        time.sleep(0.12)
    cols = ['open_time','open','high','low','close','volume',
            'close_time','quote_volume','trades','taker_buy_base',
            'taker_buy_quote','ignore']
    df = pd.DataFrame(records, columns=cols)
    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    for c in ['open','high','low','close','volume','quote_volume',
              'taker_buy_base','taker_buy_quote']:
        df[c] = df[c].astype(float)
    return df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)

START_MS = int(datetime(2024, 2, 17, tzinfo=timezone.utc).timestamp() * 1000)
END_MS   = int(datetime(2026, 4, 24, 23, 59, tzinfo=timezone.utc).timestamp() * 1000)

print('Downloading futures klines...')
df_cm   = fetch_klines(BASE_FAPI, '/fapi/v1/klines', 'BTCUSDT', START_MS, END_MS)
print('Downloading spot klines...')
df_sp   = fetch_klines(BASE_API,  '/api/v3/klines',  'BTCUSDT', START_MS, END_MS)
print('Downloading premium index klines...')
df_prem = fetch_klines(BASE_FAPI, '/fapi/v1/premiumIndexKlines', 'BTCUSDT', START_MS, END_MS)
print(f'cm={len(df_cm):,}  sp={len(df_sp):,}  prem={len(df_prem):,}')

In [ ]:
# ── 5-min feature engineering then resample to 30-min ────────────────────
# VWAP = quote_volume (U本位成交額) / volume (幣本位成交量)

def rolling_z(s, window):
    mu  = s.rolling(window, min_periods=1).mean()
    sig = s.rolling(window, min_periods=1).std().replace(0, np.nan)
    return (s - mu) / sig

# merge futures + spot + premium
df = (df_cm[['timestamp','open','high','low','close','volume','quote_volume',
              'taker_buy_base','taker_buy_quote']]
       .rename(columns={c: f'cm_{c}' for c in
           ['open','high','low','close','volume','quote_volume','taker_buy_base','taker_buy_quote']})
       .merge(df_sp[['timestamp','open','high','low','close','volume','quote_volume',
                      'taker_buy_base','taker_buy_quote']]
              .rename(columns={c: f'sp_{c}' for c in
                  ['open','high','low','close','volume','quote_volume','taker_buy_base','taker_buy_quote']}),
              on='timestamp', how='inner')
       .merge(df_prem[['timestamp','close']].rename(columns={'close':'premium_close'}),
              on='timestamp', how='inner'))

# VWAP (U-denominated formula: quote_volume / volume)
df['cm_vwap'] = df['cm_quote_volume'] / df['cm_volume']
df['sp_vwap'] = df['sp_quote_volume'] / df['sp_volume']

# CVD
df['cm_cvd_delta'] = df['cm_taker_buy_base'] - (df['cm_volume'] - df['cm_taker_buy_base'])
df['sp_cvd_delta'] = df['sp_taker_buy_base'] - (df['sp_volume'] - df['sp_taker_buy_base'])
df['cm_cvd_cum']   = df['cm_cvd_delta'].cumsum()
df['sp_cvd_cum']   = df['sp_cvd_delta'].cumsum()

# Basis
df['basis_cm_spot']     = (df['cm_close'] - df['sp_close']) / df['sp_close']
df['basis_compression'] = df['basis_cm_spot'].diff()
df['basis_from_trades'] = (df['cm_vwap'] - df['sp_vwap']) / df['sp_vwap'] * 10_000

# CVD derivatives
df['cm_cvd_change_rate']  = df['cm_cvd_cum'].pct_change().replace([np.inf,-np.inf], np.nan)
df['sp_cvd_change_rate']  = df['sp_cvd_cum'].pct_change().replace([np.inf,-np.inf], np.nan)
df['cm_cvd_acceleration'] = df['cm_cvd_change_rate'].diff()
df['sp_cvd_acceleration'] = df['sp_cvd_change_rate'].diff()
df['cvd_diff_cm_spot']    = df['cm_cvd_cum'] - df['sp_cvd_cum']

# VWAP distance from close
df['cm_vwap_dist_cm'] = (df['cm_close'] - df['cm_vwap']) / df['cm_vwap']

# resample to 30-min
df = df.set_index('timestamp')
agg_rules = {
    'cm_close':           'last', 'sp_close':          'last',
    'premium_close':      'last', 'basis_cm_spot':     'last',
    'basis_compression':  'sum',  'basis_from_trades': 'last',
    'cm_cvd_cum':         'last', 'sp_cvd_cum':        'last',
    'cm_cvd_change_rate': 'last', 'sp_cvd_change_rate':'last',
    'cm_cvd_acceleration':'last', 'sp_cvd_acceleration':'last',
    'cvd_diff_cm_spot':   'last', 'cm_vwap_dist_cm':   'last',
}
df30 = df.resample('30min', label='right', closed='right').agg(agg_rules).dropna().reset_index()

# target: futures close log return
df30['cm_logret'] = np.log(df30['cm_close'] / df30['cm_close'].shift(1))
df30 = df30.dropna().reset_index(drop=True)

# ── 12 Z-score features (all from selected_features_30m_with_formula.csv) ──
z_specs = {
    'basis_cm_spot_z3':          ('basis_cm_spot',       3),
    'basis_compression_z24':     ('basis_compression',   24),
    'basis_from_trades_z3':      ('basis_from_trades',   3),
    'cm_cvd_acceleration_z24':   ('cm_cvd_acceleration', 24),
    'cm_cvd_change_rate_z3':     ('cm_cvd_change_rate',  3),
    'cm_cvd_cum_z3':             ('cm_cvd_cum',          3),
    'cm_vwap_dist_cm_z24':       ('cm_vwap_dist_cm',     24),
    'cvd_diff_cm_spot_z3':       ('cvd_diff_cm_spot',    3),
    'premium_close_z3':          ('premium_close',       3),
    'spot_cvd_acceleration_z12': ('sp_cvd_acceleration', 12),
    'spot_cvd_change_rate_z3':   ('sp_cvd_change_rate',  3),
    'spot_cvd_cum_z3':           ('sp_cvd_cum',          3),   # ← 12th feature
}
assert len(z_specs) == 12, f'Expected 12 features, got {len(z_specs)}'

for feat, (src, w) in z_specs.items():
    df30[feat] = rolling_z(df30[src], w)

df30 = df30.dropna().reset_index(drop=True)
print(f'30-min bars: {len(df30):,}')
print(f'Z-score feature columns ({len(z_specs)}): {list(z_specs.keys())}')

## 2  Column Schema & Walk-Forward Split

In [ ]:
TIMESTAMP_COL = 'timestamp'
TARGET_COL    = 'cm_logret'

# 12 conditional / auxiliary feature columns (all Z-score normalised)
COND_COLS = [
    'basis_cm_spot_z3',          # idx 1
    'basis_compression_z24',     # idx 2
    'basis_from_trades_z3',      # idx 3
    'cm_cvd_acceleration_z24',   # idx 4
    'cm_cvd_change_rate_z3',     # idx 5
    'cm_cvd_cum_z3',             # idx 6
    'cm_vwap_dist_cm_z24',       # idx 7
    'cvd_diff_cm_spot_z3',       # idx 8
    'premium_close_z3',          # idx 9
    'spot_cvd_acceleration_z12', # idx 10
    'spot_cvd_change_rate_z3',   # idx 11
    'spot_cvd_cum_z3',           # idx 12
]
assert len(COND_COLS) == 12, f'Expected 12 conditional cols, got {len(COND_COLS)}'

INPUT_COLS = [TARGET_COL] + COND_COLS   # 1 target + 12 cond = 13 channels total
print(f'Total channels: {len(INPUT_COLS)}  (1 target + {len(COND_COLS)} conditional)')

CONTEXT_LENGTH        = 512
PREDICTION_LENGTH     = 96   # FIXED — matches ibm-research/ttm-r3 (512-96) checkpoint
PREDICTION_FILTER_LEN = 6    # trim to 6×30min = 3h at inference

SPLIT_DATE = pd.Timestamp('2025-10-01', tz='UTC')
df30['timestamp'] = pd.to_datetime(df30['timestamp'], utc=True)

train_df = df30[df30['timestamp'] < SPLIT_DATE].reset_index(drop=True)
test_df  = df30[df30['timestamp'] >= SPLIT_DATE].reset_index(drop=True)

val_split = int(len(train_df) * 0.9)
val_df    = train_df.iloc[val_split:].reset_index(drop=True)
train_df  = train_df.iloc[:val_split].reset_index(drop=True)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')

## 3  TimeSeriesPreprocessor

In [ ]:
tsp = TimeSeriesPreprocessor(
    timestamp_column=TIMESTAMP_COL,
    target_columns=INPUT_COLS,
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    scaling=True,
    scaler_type='standard',
    encode_categorical=False,
)

train_dataset = tsp.get_dataset(train_df, split='train', fewshot_fraction=1.0)
val_dataset   = tsp.get_dataset(val_df,   split='valid')
test_dataset  = tsp.get_dataset(test_df,  split='test')
print(f'train={len(train_dataset):,}  val={len(val_dataset):,}  test={len(test_dataset):,}')

## 4  Load TTM-R3 & Configure Mix-Channel Multivariate Head

In [ ]:
N_CHANNELS = len(INPUT_COLS)   # 13 (1 target + 12 conditional)

model = TinyTimeMixerForPrediction.from_pretrained(
    'ibm-research/ttm-r3',
    num_input_channels=N_CHANNELS,
    mode='mix_channel',                           # TTM-R3: cross-channel Mixer in decoder
    forecast_channel_mixing=True,                 # TTM-R3: channel mixing in forecast head
    conditional_columns=list(range(1, N_CHANNELS)),  # indices 1-12 are conditional
    prediction_length=PREDICTION_LENGTH,          # FIXED at 96
    prediction_filter_length=PREDICTION_FILTER_LEN,
    loss='pinball',
    quantiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95],
    head_dropout=0.1,
)

for name, param in model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Channels={N_CHANNELS}  Trainable={trainable:,}/{total:,} ({100*trainable/total:.1f}%)')

## 5  Fine-Tuning (Head-Only → Full Unfreeze)

In [ ]:
OUT_DIR = Path('/kaggle/working/ttm_r3_btcusdt')
OUT_DIR.mkdir(parents=True, exist_ok=True)

training_args_p1 = TrainingArguments(
    output_dir=str(OUT_DIR / 'phase1'),
    overwrite_output_dir=True,
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=1e-3,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=(DEVICE == 'cuda'),
    dataloader_num_workers=2,
    report_to='none',
    logging_steps=50,
)
trainer_p1 = Trainer(
    model=model, args=training_args_p1,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
trainer_p1.train()
print('Phase 1 complete.')

In [ ]:
for param in model.parameters():
    param.requires_grad = True

training_args_p2 = TrainingArguments(
    output_dir=str(OUT_DIR / 'phase2'),
    overwrite_output_dir=True,
    num_train_epochs=30,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=(DEVICE == 'cuda'),
    dataloader_num_workers=2,
    report_to='none',
    logging_steps=50,
)
trainer_p2 = Trainer(
    model=model, args=training_args_p2,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=7)],
)
trainer_p2.train()
print('Phase 2 complete.')

## 6  Walk-Forward Prediction

In [ ]:
from torch.utils.data import DataLoader

model.eval().to(DEVICE)
loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

QUANTILES  = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
all_preds  = []
all_actual = []

with torch.no_grad():
    for batch in loader:
        past_values   = batch['past_values'].to(DEVICE)
        future_values = batch.get('future_values')
        outputs = model(past_values=past_values)
        preds = outputs.prediction_outputs
        if preds.dim() == 4:
            preds = preds[:, :, 0, :]   # keep target channel (idx 0)
        all_preds.append(preds.cpu().numpy())
        if future_values is not None:
            all_actual.append(future_values[:, :PREDICTION_FILTER_LEN, 0].cpu().numpy())

preds_arr  = np.concatenate(all_preds,  axis=0)   # (N, 6, 7)
actual_arr = np.concatenate(all_actual, axis=0)   # (N, 6)

# inverse-scale target channel
_mu  = tsp.target_scaler.mean_[0]
_sig = tsp.target_scaler.scale_[0]
def inv_scale(arr): return arr * _sig + _mu

preds_inv  = np.stack([inv_scale(preds_arr[:, :, q]) for q in range(len(QUANTILES))], axis=-1)
actual_inv = inv_scale(actual_arr)
print(f'preds_inv: {preds_inv.shape}  actual_inv: {actual_inv.shape}')

## 7  Evaluation — Model Layer (Statistical)

In [ ]:
QIDX = {q: i for i, q in enumerate(QUANTILES)}

def pinball(y_true, y_pred, alpha):
    e = y_true - y_pred
    return np.mean(np.maximum(alpha * e, (alpha - 1) * e))

def picp(y_true, lower, upper):
    return np.mean((y_true >= lower) & (y_true <= upper))

def quantile_crossing_rate(preds_nkq):
    N, K, Q = preds_nkq.shape
    c = sum(np.sum(preds_nkq[:,:,q] > preds_nkq[:,:,q+1]) for q in range(Q-1))
    return c / (N * K * (Q - 1))

# Pinball loss per horizon step
pb_results = {}
for step in range(PREDICTION_FILTER_LEN):
    pb_results[f'Step{step+1}'] = {
        f'PB_q{int(a*100):02d}': pinball(actual_inv[:,step], preds_inv[:,step,i], a)
        for a, i in QIDX.items()}
df_pb = pd.DataFrame(pb_results).T
print('=== Pinball Loss per step ===')
print(df_pb.round(6))

# PICP + Width
for step in range(PREDICTION_FILTER_LEN):
    p80 = picp(actual_inv[:,step], preds_inv[:,step,QIDX[0.10]], preds_inv[:,step,QIDX[0.90]])
    p90 = picp(actual_inv[:,step], preds_inv[:,step,QIDX[0.05]], preds_inv[:,step,QIDX[0.95]])
    w80 = preds_inv[:,step,QIDX[0.90]] - preds_inv[:,step,QIDX[0.10]]
    print(f'Step {step+1}  PICP_80%={p80:.3f}  PICP_90%={p90:.3f}  Width_80%={np.mean(w80)*100:.4f}%')

qcr      = quantile_crossing_rate(preds_inv)
calib_50 = np.mean(actual_inv < preds_inv[:,:,QIDX[0.50]])
print(f'\nQuantile Crossing Rate: {qcr:.4f}')
print(f'Calib_0.5: {calib_50:.4f}  (target=0.5000)')

## 8  Evaluation — Strategy Layer (Trading Metrics)

In [ ]:
RISK_BUDGET = 0.25
STEP        = 0

q10 = preds_inv[:, STEP, QIDX[0.10]]
q50 = preds_inv[:, STEP, QIDX[0.50]]
q90 = preds_inv[:, STEP, QIDX[0.90]]
y   = actual_inv[:, STEP]

signal = (q10 > 0).astype(float) - (q90 < 0).astype(float)
f_star = np.clip((q50 / (np.abs(q10) + 1e-9)) * RISK_BUDGET, -1.0, 1.0)
pos    = signal * np.abs(f_star)
pnl    = pos * y

def sharpe(r, ann=17520): return np.mean(r)/(np.std(r)+1e-9)*np.sqrt(ann)
def sortino(r, ann=17520): return np.mean(r)/(np.std(r[r<0])+1e-9)*np.sqrt(ann)

cum_ret     = np.cumprod(1 + pnl)
max_dd      = ((cum_ret - np.maximum.accumulate(cum_ret)) / np.maximum.accumulate(cum_ret)).min()
active      = signal != 0
pnl_a       = pnl[active]
hit_rate    = np.mean(pnl_a > 0)
pf          = pnl_a[pnl_a>0].sum() / (np.abs(pnl_a[pnl_a<0]).sum() + 1e-9)
sig_cov     = active.mean()

print('=== Strategy Layer ===')
for k, v in [('Sharpe', sharpe(pnl)), ('Sortino', sortino(pnl)),
              ('MaxDD%', max_dd*100), ('HitRate', hit_rate),
              ('ProfitFactor', pf), ('SignalCoverage', sig_cov)]:
    print(f'  {k:16s}: {v:.4f}')

## 9  Dual-Layer Diagnostic & Visualisation

In [ ]:
picp_80 = picp(actual_inv[:,0], preds_inv[:,0,QIDX[0.10]], preds_inv[:,0,QIDX[0.90]])
_sr     = sharpe(pnl)
_cal    = float(calib_50)

print('=== Dual-Layer Diagnostic ===')
if   picp_80 >= 0.78 and _sr >= 0.5: print('✓ PICP ok + Sharpe ok: model working well.')
elif picp_80 >= 0.78 and _sr  < 0.5: print('△ PICP ok but Sharpe low: direction correct, entry timing issue.')
elif picp_80  < 0.78 and _sr  < 0.5: print('✗ PICP low + Sharpe low: tail risk underestimated. Widen stop 15-20%.')
else:                                 print('? PICP low but Sharpe ok: threshold design compensating for calibration bias.')

if abs(_cal - 0.5) < 0.02: print(f'  Calib_0.5={_cal:.4f}: q50 reliable as trend midpoint.')
else:                      print(f'  Calib_0.5={_cal:.4f}: q50 biased, recalibrate threshold.')

In [ ]:
import plotly.graph_objects as go

N_PLOT = min(500, len(actual_inv))
steps  = np.arange(1, PREDICTION_FILTER_LEN + 1)
ma = actual_inv[:N_PLOT].mean(0)
mq = {q: preds_inv[:N_PLOT,:,QIDX[q]].mean(0) for q in QUANTILES}

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[steps, steps[::-1]],
    y=np.r_[mq[0.90], mq[0.10][::-1]], fill='toself',
    fillcolor='rgba(30,144,255,0.15)', line_color='rgba(0,0,0,0)', name='80% PI'))
fig.add_trace(go.Scatter(x=np.r_[steps, steps[::-1]],
    y=np.r_[mq[0.75], mq[0.25][::-1]], fill='toself',
    fillcolor='rgba(30,144,255,0.30)', line_color='rgba(0,0,0,0)', name='50% PI'))
fig.add_trace(go.Scatter(x=steps, y=mq[0.50], name='q50',
    line=dict(color='royalblue', width=2)))
fig.add_trace(go.Scatter(x=steps, y=ma, name='Actual',
    line=dict(color='orange', width=2, dash='dot'), mode='lines+markers'))
fig.update_layout(title='TTM-R3 BTCUSDT 30m logReturn — Quantile Fan (OOS)',
    xaxis_title='Horizon Step (×30min)', yaxis_title='log Return',
    template='plotly_dark')
fig.write_image(str(OUT_DIR / 'quantile_fan.png'), width=1000, height=500)
fig.show()

In [ ]:
test_ts = test_df['timestamp'].iloc[CONTEXT_LENGTH:].reset_index(drop=True)

rows = [{'timestamp': test_ts.iloc[i] if i < len(test_ts) else np.nan,
         'actual': actual_inv[i, 0],
         **{f'q{int(a*100):02d}': preds_inv[i, 0, idx] for a, idx in QIDX.items()},
         'signal': signal[i], 'position': pos[i], 'pnl': pnl[i]}
        for i in range(len(actual_inv))]

pd.DataFrame(rows).to_csv(OUT_DIR / 'predictions_step1.csv', index=False)

pd.Series({
    'Sharpe': sharpe(pnl), 'Sortino': sortino(pnl), 'MaxDD': max_dd,
    'HitRate': hit_rate, 'ProfitFactor': pf, 'SignalCoverage': sig_cov,
    'PICP_80': picp_80, 'Calib_0.5': calib_50, 'QCR': qcr,
}).to_csv(OUT_DIR / 'summary_metrics.csv', header=['value'])
print('Saved to', OUT_DIR)